In [0]:
# ============================================================
# Notebook: 03_generate_dimensions
# Purpose : Generate synthetic account and customer dimensions
#           from PaySim account IDs.
# ============================================================

from pyspark.sql.functions import col, lit, rand, when, current_date, date_sub, concat
from pyspark.sql.functions import monotonically_increasing_id

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

SILVER_TXN_TABLE = "data_engineering.silver_layer.finshield_silver"

DIM_ACCOUNTS_TABLE = "data_engineering.silver_layer.dim_accounts"
DIM_CUSTOMERS_TABLE = "data_engineering.silver_layer.dim_customers"

# ------------------------------------------------------------
# 2. Read Silver transaction data
# ------------------------------------------------------------

txn_df = spark.table(SILVER_TXN_TABLE)

# ------------------------------------------------------------
# 3. Extract unique customer/account IDs from origin accounts
#    We use origin_account_id because it represents sender accounts.
# ------------------------------------------------------------

unique_accounts_df = txn_df.select(
    col("origin_account_id").alias("account_id")
).distinct()

print("Unique origin accounts:")
print(unique_accounts_df.count())

# ------------------------------------------------------------
# 4. Generate Account Dimension
#    This represents account-level information.
# ------------------------------------------------------------

dim_accounts_df = unique_accounts_df.withColumn(
    "account_type",
    when(rand() < 0.50, "SAVINGS")
    .when(rand() < 0.80, "CURRENT")
    .otherwise("BUSINESS")
).withColumn(
    "status",
    when(rand() < 0.90, "ACTIVE")
    .when(rand() < 0.97, "SUSPENDED")
    .otherwise("CLOSED")
).withColumn(
    "credit_score",
    (rand() * 550 + 300).cast("int")
).withColumn(
    "kyc_verified",
    when(rand() < 0.85, lit(True)).otherwise(lit(False))
).withColumn(
    "opened_date",
    date_sub(current_date(), (rand() * 3000).cast("int"))
).withColumn(
    "is_deleted",
    lit(False)
).withColumn(
    "created_at",
    current_date()
).withColumn(
    "updated_at",
    current_date()
)

# ------------------------------------------------------------
# 5. Write Account Dimension
# ------------------------------------------------------------

dim_accounts_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(DIM_ACCOUNTS_TABLE)

print("dim_accounts created successfully.")

# ------------------------------------------------------------
# 6. Generate Customer Dimension
#    This represents customer profile information.
#    Later we will use this for SCD Type 2.
# ------------------------------------------------------------

dim_customers_df = unique_accounts_df.withColumn(
    "customer_id",
    monotonically_increasing_id()
).withColumn(
    "full_name",
    concat(lit("Customer_"), col("customer_id"))
).withColumn(
    "country",
    lit("India")
).withColumn(
    "risk_tier",
    when(rand() < 0.70, "LOW")
    .when(rand() < 0.90, "MEDIUM")
    .otherwise("HIGH")
).withColumn(
    "address",
    concat(lit("Address_"), col("customer_id"))
).withColumn(
    "effective_start_date",
    current_date()
).withColumn(
    "effective_end_date",
    lit(None).cast("date")
).withColumn(
    "is_current",
    lit(True)
).withColumn(
    "created_at",
    current_date()
).withColumn(
    "updated_at",
    current_date()
)

# ------------------------------------------------------------
# 7. Write Customer Dimension
# ------------------------------------------------------------

dim_customers_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(DIM_CUSTOMERS_TABLE)

print("dim_customers created successfully.")

# ------------------------------------------------------------
# 8. Validation
# ------------------------------------------------------------

print("Account dimension count:")
spark.sql(f"""
SELECT COUNT(*) AS account_count
FROM {DIM_ACCOUNTS_TABLE}
""").display()

print("Customer dimension count:")
spark.sql(f"""
SELECT COUNT(*) AS customer_count
FROM {DIM_CUSTOMERS_TABLE}
""").display()

print("Sample accounts:")
spark.table(DIM_ACCOUNTS_TABLE).display()

print("Sample customers:")
spark.table(DIM_CUSTOMERS_TABLE).display()